In [1]:
import pandas as pd
import numpy as np


In [9]:
# Training data
y_train = pd.read_csv("original/water_quality_training_dataset.csv")
x_train_All = pd.read_csv("Data/training_merged.csv")
print("Training Response Features Shape: ", y_train.shape)
print("Training Features Shape: ", x_train_All.shape)
landsat_only_train = pd.read_csv("original/landsat_features_training.csv")
print(landsat_only_train.shape)

# Validation data
submission_template = pd.read_csv("Data/submission_template.csv")
x_test_All = pd.read_csv("Data/validation_merged.csv")


print(landsat_only_train.head())

Training Response Features Shape:  (9319, 6)
Training Features Shape:  (8985, 86)
(9319, 9)
    Latitude  Longitude Sample Date      nir    green   swir16   swir22  \
0 -28.760833  17.730278  02-01-2011  11190.0  11426.0   7687.5   7645.0   
1 -26.861111  28.884722  03-01-2011  17658.5   9550.0  13746.5  10574.0   
2 -26.450000  28.085833  03-01-2011  15210.0  10720.0  17974.0  14201.0   
3 -27.671111  27.236944  03-01-2011  14887.0  10943.0  13522.0  11403.0   
4 -27.356667  27.286389  03-01-2011  16828.5   9502.5  12665.5   9643.0   

       NDMI     MNDWI  
0  0.185538  0.195595  
1  0.124566 -0.180134  
2 -0.083293 -0.252805  
3  0.048048 -0.105416  
4  0.141147 -0.142683  


In [13]:
test_concat = y_train.merge(
    landsat_only_train.reset_index(), 
    on=["Latitude", "Longitude"], 
    how="left"
)

In [19]:
print(y_train.shape)
y_train[["Longitude","Latitude"]].nunique()

(9319, 6)


Longitude    161
Latitude     161
dtype: int64

In [20]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'.')

from panel_model import (
    PanelWaterQualityModel,
    spatial_cv,
    compute_site_means,
    TIME_VARIANT_COLS,
    TARGETS
)
print("imports OK")

imports OK


In [23]:
train = pd.read_csv("Data/training_merged.csv")
val = pd.read_csv("Data/validation_merged.csv")
submission_template = pd.read_csv("Data/submission_template.csv")

print(f"Train: {train.shape} | {train.groupby(["Latitude","Longitude"]).ngroups} sites")
print(f"Validation: {val.shape} | {val.groupby(["Latitude","Longitude"]).ngroups} sites")

Train: (8985, 86) | 162 sites
Validation: (200, 83) | 24 sites


In [24]:
def add_geology_flags(df):
    def flags(name):
        n = str(name).lower()
        is_karoo = int(any(k in n for k in [
            'karoo','beaufort','ecca','dwyka','molteno','elliot','clarens',
            'vryheid','volksrust','tarkastad','fort brown','waterford',
            'prince albert','emakwezini','pietermaritzburg','dolerite',
        ]))
        is_cape = int(any(k in n for k in [
            'cape supergroup','witteberg','bokkeveld','table mountain',
            'nardouw','ceres subgroup','natal group',
        ]))
        return pd.Series({
            'is_karoo_supergroup': is_karoo,
            'is_cape_supergroup':  is_cape,
            'is_beaufort_group':   int(any(k in n for k in ['beaufort','tarkastad','adelaide','molteno','elliot','clarens','emakwezini'])),
            'is_ecca_group':       int(any(k in n for k in ['ecca','volksrust','vryheid','fort brown','waterford','prince albert','pietermaritzburg'])),
            'is_dwyka_group':      int('dwyka' in n),
            'is_karoo_dolerite':   int('dolerite' in n),
        })
    flags_df = df['macrostrat_name'].apply(flags)
    return pd.concat([df, flags_df], axis=1)

train = add_geology_flags(train)
val = add_geology_flags(val)

print("Geology flag coverage - train vs. val:")
for col in ['is_karoo_supergroup','is_cape_supergroup','is_beaufort_group',
            'is_ecca_group','is_dwyka_group','is_karoo_dolerite']:
    print(f"  {col:25s}: train={train[col].mean():.2f}  val={val[col].mean():.2f}")

Geology flag coverage - train vs. val:
  is_karoo_supergroup      : train=0.48  val=0.78
  is_cape_supergroup       : train=0.09  val=0.12
  is_beaufort_group        : train=0.16  val=0.61
  is_ecca_group            : train=0.19  val=0.01
  is_dwyka_group           : train=0.08  val=0.00
  is_karoo_dolerite        : train=0.04  val=0.16


In [25]:
for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  {target}")
    print('='*55)
    spatial_cv(train, target=target, n_splits=5)


  Total Alkalinity


ValueError: Input X contains NaN.
Ridge does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values